# Two-head GapFinder experiment

This notebook tests a shared Qwen transformer with two outputs:

1. Regression: predicted normalized gap `d_hat`.
2. Detection: probability `P(d > theta)`.

It reuses the exact 20k dataset and deterministic 70/15/15 split from `gap_finder_predictability_experiment.ipynb`, allowing a fair comparison with regression-only GapFinder. No PPO is run here.

In [ ]:
import json
import logging
from pathlib import Path

import numpy as np
import pandas as pd

import functions
from Datasets.dataset_gap_finder import DatasetGapFinder
from Models.lora import LoRASettings
from Models.model_two_head_gap_finder import TwoHeadGapFinder
from Trainers.trainer_two_head_gap_finder import (
    TwoHeadGapFinderTrainer,
    TwoHeadGapFinderTrainingConfig,
    compute_two_head_report,
)

logging.getLogger().setLevel(logging.INFO)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("httpx").setLevel(logging.WARNING)

## 1. Load the existing calibration and identical data split

Run `gap_finder_predictability_experiment.ipynb` through its data-collection cell first if these files do not exist.

In [ ]:
SOURCE_ROOT = Path("outputs/gap_finder_predictability")
OUTPUT_ROOT = Path("outputs/two_head_gap_finder")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42

calibration_path = SOURCE_ROOT / "calibration.json"
dataset_path = SOURCE_ROOT / "all_20k.json"
if not calibration_path.is_file() or not dataset_path.is_file():
    raise FileNotFoundError(
        "Run gap_finder_predictability_experiment.ipynb through data collection first"
    )

gap_calibration = functions.GapCalibration.load(calibration_path)
gap_dataset = DatasetGapFinder.load(dataset_path)
train_dataset, validation_dataset, test_dataset = gap_dataset.split_three_way(
    train_size=0.70,
    validation_size=0.15,
    test_size=0.15,
    random_state=RANDOM_STATE,
)
assert (len(train_dataset), len(validation_dataset), len(test_dataset)) == (14_000, 3_000, 3_000)
{
    "theta": gap_calibration.theta,
    "train": len(train_dataset),
    "validation": len(validation_dataset),
    "test": len(test_dataset),
}

In [ ]:
def tail_summary(dataset):
    gaps = np.asarray([row["labels"] for row in dataset.dataset])
    positives = int((gaps > gap_calibration.theta).sum())
    return {
        "examples": len(gaps),
        "d_gt_theta": positives,
        "positive_fraction": positives / len(gaps),
    }

pd.DataFrame(
    {
        "train": tail_summary(train_dataset),
        "validation": tail_summary(validation_dataset),
        "test": tail_summary(test_dataset),
    }
).T

## 2. Train the shared two-output model

The objective is `tail-weighted MSE + lambda * class-weighted BCE`. The BCE positive weight is calculated automatically from the training split. The best epoch is selected by validation PR-AUC.

In [ ]:
MODEL_NAME = "Qwen/Qwen3-0.6B"
HIGH_GAP_WEIGHT = 5.0
DETECTOR_LOSS_WEIGHT = 1.0
run_directory = (
    OUTPUT_ROOT
    / f"alpha={HIGH_GAP_WEIGHT:g}_lambda={DETECTOR_LOSS_WEIGHT:g}"
)

two_head_model = TwoHeadGapFinder(
    MODEL_NAME,
    theta=gap_calibration.theta,
    gap_finder_id=gap_dataset.id,
    source_policy="Qwen/Qwen3-0.6B",
)
two_head_config = TwoHeadGapFinderTrainingConfig(
    output_dir=str(run_directory),
    theta=gap_calibration.theta,
    epochs=3.0,
    batch_size=16,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    learning_rate=2e-5,
    max_length=512,
    high_gap_weight=HIGH_GAP_WEIGHT,
    detector_loss_weight=DETECTOR_LOSS_WEIGHT,
    lora_settings=LoRASettings(),
)
two_head_trainer = TwoHeadGapFinderTrainer(two_head_model, two_head_config)
hf_trainer = two_head_trainer.train(train_dataset, validation_dataset)
validation_metrics = two_head_trainer.evaluate(
    hf_trainer, validation_dataset, prefix="validation"
)
validation_metrics

## 3. Evaluate once on the untouched test split

In [ ]:
test_prompts = [row["prompt"] for row in test_dataset.dataset]
test_answers = [row["answer"] for row in test_dataset.dataset]
actual_test_gaps = np.asarray([row["labels"] for row in test_dataset.dataset])
predicted_test_gaps, detection_probabilities = two_head_model.predict(
    test_prompts,
    test_answers,
    batch_size=16,
    max_length=two_head_config.max_length,
)
predicted_test_gaps = np.asarray(predicted_test_gaps)
detection_probabilities = np.asarray(detection_probabilities)

test_report = compute_two_head_report(
    actual_test_gaps,
    predicted_test_gaps,
    detection_probabilities,
    theta=gap_calibration.theta,
)
zero_mse = float(np.mean(actual_test_gaps ** 2))
zero_mae = float(np.mean(np.abs(actual_test_gaps)))
test_report["mse_improvement_over_zero"] = 1.0 - test_report["mse"] / zero_mse
test_report["mae_improvement_over_zero"] = 1.0 - test_report["mae"] / zero_mae
test_report.update(
    {
        "theta": gap_calibration.theta,
        "high_gap_weight": HIGH_GAP_WEIGHT,
        "detector_loss_weight": DETECTOR_LOSS_WEIGHT,
    }
)
report_path = run_directory / "test_report.json"
report_path.write_text(json.dumps(test_report, indent=2, sort_keys=True) + "\n")
pd.DataFrame([test_report]).T.rename(columns={0: "test_value"})

## 4. Compare with the regression-only experiment

The regression-only report is loaded when available. Its threshold metrics come from thresholding its gap prediction, whereas the two-head columns use the dedicated detector output.

In [ ]:
regression_report_path = (
    SOURCE_ROOT / f"id={gap_dataset.id}" / "test_report.json"
)
regression_report = (
    json.loads(regression_report_path.read_text())
    if regression_report_path.is_file()
    else {}
)
comparison = pd.DataFrame(
    [
        {
            "model": "regression_only",
            "mse": regression_report.get("mse"),
            "r2": regression_report.get("r2"),
            "mae_d_gt_theta": regression_report.get("mae_d_gt_theta"),
            "precision": regression_report.get("precision_d_gt_theta"),
            "recall": regression_report.get("recall_d_gt_theta"),
            "f1": regression_report.get("f1_d_gt_theta"),
            "pr_auc": None,
        },
        {
            "model": "two_head",
            "mse": test_report["mse"],
            "r2": test_report["r2"],
            "mae_d_gt_theta": test_report["mae_d_gt_theta"],
            "precision": test_report["detector_precision"],
            "recall": test_report["detector_recall"],
            "f1": test_report["detector_f1"],
            "pr_auc": test_report["detector_pr_auc"],
        },
    ]
).set_index("model")
comparison

## 5. Inspect tail examples and detector errors

In [ ]:
test_examples = pd.DataFrame(
    {
        "prompt": test_prompts,
        "answer": test_answers,
        "actual_gap": actual_test_gaps,
        "predicted_gap": predicted_test_gaps,
        "p_d_gt_theta": detection_probabilities,
        "actual_detector": actual_test_gaps > gap_calibration.theta,
        "predicted_detector": detection_probabilities >= 0.5,
    }
)
test_examples["detector_correct"] = (
    test_examples["actual_detector"] == test_examples["predicted_detector"]
)
test_examples.sort_values(
    ["actual_detector", "actual_gap"], ascending=[False, False]
).head(30)

## Optional next sweep

After this run, repeat from a freshly initialized model with `DETECTOR_LOSS_WEIGHT` in `{0.25, 0.5, 1.0, 2.0}`. Do not choose lambda using test results; select it using validation PR-AUC while ensuring validation MSE does not collapse, then evaluate the chosen configuration once on test.